# Sigiriya SinLlama evidence handoff v2

Keep the existing evidence JSON for a fair comparison. From the Windows project directory, regenerate only the runtime ZIP.

```powershell
.\.venv\Scripts\python.exe -B -X utf8 -m scripts.sinllama_handoff --runtime-only
```

Use **Runtime > Change runtime type > T4 GPU**. Run the notebook in a fresh runtime. Upload `sigiriya_sinllama_evidence.json` and `sigiriya_sinllama_runtime_v2.zip` from `results/generation/`. Neo4j and E5 stay on Windows.

This [Sinhala base completion model](https://huggingface.co/SAWithanage/SinLlama-Llama-3-8B-Merged) uses plain completion prompts. The verifier still checks complete, ordered copies of the selected excerpts.


## 1. Check the GPU and install inference dependencies

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")
print(torch.cuda.get_device_name(0), torch.__version__)


Tesla T4 2.14.1+cu130


In [2]:
%pip install -q transformers==4.57.1 accelerate==1.10.1 bitsandbytes==0.48.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Upload the exported evidence and runtime ZIP

In [3]:
import json
import sys
from pathlib import Path
from zipfile import ZipFile

from google.colab import files

uploaded = files.upload()
evidence_name = "sigiriya_sinllama_evidence.json"
runtime_name = "sigiriya_sinllama_runtime_v2.zip"
if not {evidence_name, runtime_name} <= uploaded.keys():
    raise ValueError("Upload both files produced by the local exporter.")

runtime = Path("/content/sigiriya_sinllama_runtime")
expected = {
    "src/__init__.py", "src/config.py", "src/generation.py", "src/prompting.py",
    "scripts/sinllama_handoff.py",
}
with ZipFile(runtime_name) as archive:
    if set(archive.namelist()) != expected:
        raise ValueError("Unexpected runtime files. Re-export from the local project.")
    archive.extractall(runtime, members=sorted(expected))
sys.path.insert(0, str(runtime))
payload = json.loads(Path(evidence_name).read_text(encoding="utf-8-sig"))
print(f"Loaded {len(payload['questions'])} questions.")


Saving sigiriya_sinllama_evidence.json to sigiriya_sinllama_evidence.json
Saving sigiriya_sinllama_runtime_v2.zip to sigiriya_sinllama_runtime_v2.zip
Loaded 9 questions.


## 3. Generate and display each question immediately

Leave **QUESTION_NUMBER** at **0** for all exported questions. Enter **1?9** for one of the standard nine questions. Choose a **PROFILE** and rerun the cell to test a different setting. The first eligible question loads the model in 4-bit NF4 on the T4. The display shortens only the raw proposal preview. The complete decoded completion stays in JSON.


In [4]:
from importlib.metadata import version

from scripts.sinllama_handoff import (
    COMPARISON_PROFILES, format_question_result, generate_batch, summarize_generation, write_json,
)
from src.generation import SinLlamaGenerator

QUESTION_NUMBER = 0  # @param {type:"integer"}
PROFILE = "source_qa"  # @param ["source_qa", "sinhala_qa", "short_grounding", "source_qa_penalty_1_15"]

if "generator" not in globals():
    generator = SinLlamaGenerator(local_files_only=False)
generator.config = COMPARISON_PROFILES[PROFILE]
runtime_info = {
    "gpu": torch.cuda.get_device_name(0),
    "versions": {pkg: version(pkg) for pkg in ("torch", "transformers", "accelerate", "bitsandbytes")},
}
output = generate_batch(
    payload, generator, question_number=QUESTION_NUMBER or None,
    on_result=lambda number, row: print(format_question_result(number, row), flush=True),
)
output["runtime"] = runtime_info
output["profile"] = PROFILE
output["summary"] = summarize_generation(output["results"])
suffix = f"_q{QUESTION_NUMBER:02d}" if QUESTION_NUMBER else ""
result_path = Path(f"sigiriya_sinllama_results_v2{suffix}.json")
write_json(result_path, output)
print(output["summary"])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/331 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/19.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/737 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

... (10 outputs left out)

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/170 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


QUESTION 01:
සීගිරි බිතුසිතුවම්වල කාන්තා රූප ලෙස නිරූපණය කර ඇත්තේ කවුරුන්ද?

SINLLAMA GENERATED ANSWER:
සීගිරි බිතුසිතුවම්වල කාන්තා රූප ලෙස නිරූපණය කර ඇත්තේ අප්සරාවන් හෝ දිව්‍ය කාන්තාවන් ය.මෙම සිතුවම්වල කාන්තා රූප දැකගත හැකි අතර ඒවා අප්සරාවන් හෝ දිව්‍ය කාන්තාවන් ලෙස අර්ථකථනය කර ඇත.මෙම සිතුවම්වල කාන්තා රූප දැකගත හැකි අතර ඒවා අප්සරාවන් හෝ දිව්‍ය කාන්තාවන් ලෙස අර්ථකථනය කර ඇත.මෙම සිතුවම්වල කාන්තා රූප දැකගත හැකි අතර ඒවා අප්සරාවන් හෝ දිව්‍ය කාන්තාවන් ලෙස අර්ථකථනය කර ඇත.

FINAL VERIFIED ANSWER:
මෙම සිතුවම්වල කාන්තා රූප දැකගත හැකි අතර ඒවා අප්සරාවන් හෝ දිව්‍ය කාන්තාවන් ලෙස අර්ථකථනය කර ඇත.

STATUS:
fallback (repetition_or_bad_format)

SOURCE CHUNKS:
sigiriya_004

QUESTION 02:
සීගිරි බිතුසිතුවම් ගැන කියන්න.

SINLLAMA GENERATED ANSWER:
සීගිරි බිතුසිතුවම් ගැන කියන්න.

FINAL VERIFIED ANSWER:
දැනට ඉතිරිව ඇති බිතුසිතුවම් භූමියෙන් මීටර් 100කට වඩා ඉහළින් පිහිටි පර්වතයේ කොටසක දැකගත හැකිය. ඒවා වෙත ළඟාවීම සඳහා සර්පිලාකාර පඩිපෙළක් භාවිත කරයි. මෙම සිතුවම්වල කාන්තා රූප දැකගත හැකි අතර ඒවා අප්සරාවන් හෝ දිව්‍ය ක

## 4. Download the v2 result

In [5]:
files.download(str(result_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Optional full prompt comparison

Set **RUN_COMPARISON** to true and run the next cell. Set **UPLOAD_BASELINE** to true to include the original `sigiriya_sinllama_results.json`. The notebook checks that it used the same questions and selected excerpts. Each v2 profile uses the same packet, verifier, token limits, and greedy decoding. The first three profiles vary only the prompt. The fourth keeps the reference prompt and changes only the repetition penalty. The historical run differs in multiple settings, so use it as a reference rather than as a controlled contrast.

Review the saved proposals against their sources and fill the manual fields for relevance, factual correctness, topic completeness, and Sinhala readability. Inspect possible paraphrases separately and keep the verified final answers as exported.


In [6]:
from scripts.sinllama_handoff import compare_generation

RUN_COMPARISON = True  # @param {type:"boolean"}
UPLOAD_BASELINE = False  # @param {type:"boolean"}

if RUN_COMPARISON:
    baseline = None
    if UPLOAD_BASELINE:
        uploaded_baseline = files.upload()
        baseline_bytes = uploaded_baseline["sigiriya_sinllama_results.json"]
        baseline = json.loads(baseline_bytes.decode("utf-8-sig"))

    def show_comparison(profile, number, row):
        print(f"=== {profile} ===")
        print(format_question_result(number, row), flush=True)

    comparison = compare_generation(
        payload, generator, baseline=baseline, on_result=show_comparison,
    )
    comparison["runtime"] = runtime_info
    comparison_path = Path("sinllama_prompt_comparison.json")
    write_json(comparison_path, comparison)
    for profile, run in comparison["runs"].items():
        print(profile, run["summary"])
    files.download(str(comparison_path))
